In [1]:
%load_ext autoreload
%autoreload 2

from langchain_core.runnables import history

from llm_sdk import LLM_SDK
from openai import OpenAI, api_key
from dotenv import load_dotenv
from typing import Dict
import os
import json

class learnAgent:   
    def __init__(self, llm_client: LLM_SDK):
        self.llm = llm_client
        self.trajectory:list[Dict[str,str]] = []
    
    def plan(self,task: str) -> str:
        sys_prompt = """
你是任务规划专家，将复杂任务拆解为串行执行步骤。
严格只返回JSON，不要额外解释，格式：
{
    "steps": ["步骤1","步骤2","步骤3"]
}
"""
        messages = [
            {"role":"system","content":sys_prompt},
            {"role":"user","content":f"目标任务：{task}"}
        ]
        raw_txt = self.llm.think(messages,temperature=0,stream=False)
        try:
            plan = json.loads(raw_txt)
            # 让规划阶段可见，避免误以为没走 plan
            print(f"\n📋 规划完成，步骤列表：{plan.get('steps', [])}")
            return plan
        except Exception as e:
            print(f"❌ 解析JSON时发生错误: {e}\n原始文本：{raw_txt}")
            return {"steps": []}
    def execute(self,step: str, global_task: str) -> str:
        """ 
        执行步骤
        """ 
        # 修复：补 f 前缀，让 {global_task} / {step} 真正被替换，而不是字面占位符
        system_prompt = f"""
当前总目标：{global_task}
需要执行子步骤：{step}
请完成该步骤，输出执行结果。
"""

        messages = [
            {"role":"system","content":system_prompt},
            {"role":"user","content":f" 开始执行"}
        ]
        result = self.llm.think(messages,temperature=0,stream=False)
        self.trajectory.append({
            "step":step,
            "result":result
        })
        return result
    
    def reflect(self,global_task: str) -> dict:
        """ 
        反思执行结果
        """ 

        system_prompt = """
你是自检评审专家。根据任务目标与已执行记录判断：
1. 当前信息是否足够完成最终目标
2. 是否需要新增步骤继续执行。
严格只返回JSON，不要额外解释，格式：
{
    "finished": true/false,
    "suggestion": "说明理由，如果未完成给出新增步骤"
}
"""
        history_txt=json.dumps(self.trajectory,ensure_ascii=False,indent=2)
        messages = [
            {"role":"system","content":system_prompt},
            {"role":"user","content":f" 反思执行结果，任务：{global_task}，执行历史：{history_txt}"}
        ]
        raw_txt = self.llm.think(messages,temperature=0,stream=False)
        try:
            reflect = json.loads(raw_txt)
            return reflect
        except Exception as e:
            print(f"❌ 解析JSON时发生错误: {e}\n原始文本：{raw_txt}")
            return {"reflect": ""}
    
    def run(self,task: str):
        """ 
        运行任务
        """ 
        plan = self.plan(task)
        steplist = plan.get("steps",[])

        max_round = 10
        round_count = 0

        while round_count < max_round:
            round_count += 1
            print(f"\n【第{round_count}轮执行】")
            for step in steplist:
                print(f"执行步骤：{step}")
                res = self.execute(step,task)
                print(f"执行结果：{res}\n")


            reflect_res = self.reflect(task)
            if reflect_res["finished"]:
                break
            steplist = [reflect_res["suggestion"]]
            print(f"新增步骤：{steplist}")
        else:
            print("达到最大轮次限制，强制终止")
        print("任务执行完成")
        return self.trajectory

load_dotenv()

api_key = os.getenv("DASHSCOPE_API_KEY")   
base_url = os.getenv("DASHSCOPE_API_URL_RESPONSE")

if __name__ == "__main__":
    
    llm_client = LLM_SDK(api_key,base_url)
    agent = learnAgent(llm_client)
    task = "统计 1~100 所有数字的总和，并给出计算过程"
    agent.run(task)



📋 规划完成，步骤列表：['识别任务为连续整数求和，选用等差数列求和公式 S = n×(首项+末项)/2', '确定参数：项数n=100，首项=1，末项=100', '代入公式执行计算：S = 100×(1+100)/2 = 5050', '按逻辑顺序排版并输出完整的计算过程与最终结果']

【第1轮执行】
执行步骤：识别任务为连续整数求和，选用等差数列求和公式 S = n×(首项+末项)/2
执行结果：已按子步骤要求执行，具体过程与结果如下：

**1. 任务识别**  
该问题属于连续正整数求和问题，符合等差数列特征（公差为1）。

**2. 公式选用**  
采用等差数列求和公式：  
`S = n × (首项 + 末项) / 2`

**3. 参数代入**  
- 项数 `n = 100`  
- 首项 = `1`  
- 末项 = `100`  

**4. 计算过程**  
```
S = 100 × (1 + 100) / 2
  = 100 × 101 / 2
  = 10100 / 2
  = 5050
```

**执行结果**  
1~100 所有数字的总和为 **5050**。

执行步骤：确定参数：项数n=100，首项=1，末项=100
执行结果：【步骤执行结果】
参数已确认：项数 `n = 100`，首项 `a₁ = 1`，末项 `aₙ = 100`。

---
【完整计算过程】（按总目标补充）
该序列为公差为 1 的等差数列，使用等差数列求和公式：
$$S_n = \frac{n \times (a_1 + a_n)}{2}$$

代入已确定的参数：
$$S_{100} = \frac{100 \times (1 + 100)}{2}$$
$$= \frac{100 \times 101}{2}$$
$$= 50 \times 101$$
$$= 5050$$

【最终结果】
1~100 所有数字的总和为 **5050**。

执行步骤：代入公式执行计算：S = 100×(1+100)/2 = 5050
执行结果：已按指定子步骤执行计算：

**代入公式**：`S = 100 × (1 + 100) / 2`  
**计算过程**：
1. 先计算括号内：`1 + 100 = 101`
2. 再与项数相乘：`100 × 101 = 10